# 04 — Politik-Query (platzsparend, mit Rank-Chunks zum Abschneiden)

Liest [`data/accounts.csv`](../data/accounts.csv) und baut **eine** Brandwatch-Query.

**Filter:**
- `channel ∈ {x, instagram, facebook}` — keine Websites, kein YouTube/TikTok.
- `category ∈ {Politician, Organisation}`.
- `label != "Behörde"`.

**Struktur** (OR-verknüpft, platzsparend — Handles auf einer Zeile, ein `\n` pro Block):

1. **X** — ein Block mit allen X-Handles (kein Ranking-Split: X hat im Datensatz keine Ranks).
2. **Facebook**
   1. Block *ohne Ranking* (oben).
   2. Danach **10 Rank-Chunks** in aufsteigender Rank-Reihenfolge. Chunk-Breite =
      `ceil(max_rank / 10)`, d. h. Chunk 1 = Rank 1…W, Chunk 2 = W+1…2W, …
3. **Instagram** — gleiche Logik wie Facebook (*ohne Ranking* zuerst, dann 10 Chunks).

So kannst du von unten her (= höchste Ranks = weniger wichtige Handles) Blöcke
wegschneiden, bis die Query unter das Brandwatch-Limit von 100.000 Zeichen
(siehe [`brandwatch_query_syntax.md`](./brandwatch_query_syntax.md) #14) passt.

**Output:** `output/queries/politics_query.txt` (eine Datei).

In [1]:
import math
import os

import pandas as pd

if os.path.basename(os.getcwd()) == "scripts":
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()

DATA_DIR    = os.path.join(PROJECT_ROOT, "data")
QUERIES_DIR = os.path.join(PROJECT_ROOT, "output", "queries")
ACCOUNTS_CSV = os.path.join(DATA_DIR, "accounts.csv")
OUTPUT_FILE  = os.path.join(QUERIES_DIR, "politics_query.txt")

ALLOWED_CHANNELS   = ["x", "instagram", "facebook"]
ALLOWED_CATEGORIES = ["Politician", "Organisation"]
EXCLUDED_LABELS    = {"Behörde"}

LANGUAGE_FILTER = "language:de"
N_RANK_CHUNKS   = 10

os.makedirs(QUERIES_DIR, exist_ok=True)

## 1. Daten laden + filtern

In [2]:
accounts = pd.read_csv(ACCOUNTS_CSV)

base = (
    accounts[
        accounts["channel"].isin(ALLOWED_CHANNELS)
        & accounts["category"].isin(ALLOWED_CATEGORIES)
        & ~accounts["label"].isin(EXCLUDED_LABELS)
    ]
    .dropna(subset=["handle"])
    .drop_duplicates(subset=["channel", "handle"])
    .copy()
)

print(f"Total nach Filter: {len(base)}")
for ch in ALLOWED_CHANNELS:
    sub = base[base["channel"] == ch]
    r = sub["rank"]
    print(
        f"  {ch:>10s}: {len(sub):>5} total | ranked={int(r.notna().sum()):>5} | "
        f"unranked={int(r.isna().sum()):>5} | rank-range={r.min()}–{r.max()}"
    )

Total nach Filter: 6365
           x:  1781 total | ranked=    0 | unranked= 1781 | rank-range=nan–nan
   instagram:  2200 total | ranked= 1731 | unranked=  469 | rank-range=8.0–3151.0
    facebook:  2384 total | ranked= 1373 | unranked= 1011 | rank-range=1.0–3432.0


## 2. Helper

In [3]:
def bw_author(handle: str) -> str:
    h = str(handle).strip().replace('"', '\\"')
    return f'author:"{h}"'


def or_line(handles) -> str:
    """Alle Handles in einer Zeile, mit ' OR ' getrennt, geklammert."""
    return "(" + " OR ".join(bw_author(h) for h in handles) + ")"


def block(comment: str, handles) -> str:
    """Ein Abschnitt: Kommentar-Zeile + OR-Zeile."""
    return f"<<< {comment} — {len(handles)} Handles >>>\n{or_line(handles)}"


def sort_handles(series: pd.Series):
    return series.sort_values(key=lambda s: s.str.lower()).tolist()


def rank_chunk_edges(max_rank: int, n: int = N_RANK_CHUNKS):
    """Liefert n (start, end)-Tupel, die [1, max_rank] gleichmäßig in n Chunks teilen.

    Chunk-Breite = ceil(max_rank / n). Beispiel max_rank=3432, n=10 → Breite 344,
    Chunks: (1,344), (345,688), …, (3097,3432).
    """
    if max_rank <= 0:
        return []
    width = math.ceil(max_rank / n)
    edges = []
    start = 1
    while start <= max_rank:
        end = min(start + width - 1, max_rank)
        edges.append((start, end))
        start = end + 1
    return edges


def build_ranked_blocks(platform_label: str, df_platform: pd.DataFrame) -> list[str]:
    """
    Baut fuer eine Plattform die Blöcke:
      1) `ohne Ranking`  (falls vorhanden, immer ganz oben)
      2) Rank-Chunks aufsteigend (niedrigster Rank zuerst)
    """
    blocks: list[str] = []

    unranked = df_platform[df_platform["rank"].isna()]
    if len(unranked) > 0:
        blocks.append(block(f"{platform_label} — ohne Ranking", sort_handles(unranked["handle"])))

    ranked = df_platform[df_platform["rank"].notna()].copy()
    if len(ranked) == 0:
        return blocks

    ranked["rank"] = ranked["rank"].astype(int)
    max_rank = int(ranked["rank"].max())
    for start, end in rank_chunk_edges(max_rank):
        sub = ranked[(ranked["rank"] >= start) & (ranked["rank"] <= end)].sort_values("rank")
        if len(sub) == 0:
            continue
        blocks.append(block(f"{platform_label} — Rank {start}–{end}", sub["handle"].tolist()))

    return blocks

## 3. Blöcke bauen (X → Facebook → Instagram)

In [4]:
all_blocks: list[str] = []

# --- 1) X: EIN Block mit allen Handles --------------------------------------
x_df = base[base["channel"] == "x"]
if len(x_df) > 0:
    all_blocks.append(block("X (Twitter)", sort_handles(x_df["handle"])))
    print(f"X:         1 Block  ({len(x_df)} Handles)")

# --- 2) Facebook: ohne-Ranking + 10 Rank-Chunks -----------------------------
fb_df = base[base["channel"] == "facebook"]
fb_blocks = build_ranked_blocks("Facebook", fb_df)
all_blocks.extend(fb_blocks)
print(f"Facebook:  {len(fb_blocks)} Blöcke ({len(fb_df)} Handles)")

# --- 3) Instagram: ohne-Ranking + 10 Rank-Chunks ----------------------------
ig_df = base[base["channel"] == "instagram"]
ig_blocks = build_ranked_blocks("Instagram", ig_df)
all_blocks.extend(ig_blocks)
print(f"Instagram: {len(ig_blocks)} Blöcke ({len(ig_df)} Handles)")

print(f"\nBlöcke gesamt: {len(all_blocks)}")

X:         1 Block  (1781 Handles)
Facebook:  11 Blöcke (2384 Handles)
Instagram: 11 Blöcke (2200 Handles)

Blöcke gesamt: 23


## 4. Gesamt-Query zusammensetzen

Format (platzsparend):
```
<<< Politik-Query — N Handles in M Blöcken >>>
(language:de AND (
<<< X (Twitter) — N Handles >>>
(author:"..." OR author:"..." OR …)
OR
<<< Facebook — ohne Ranking — N Handles >>>
(...)
OR
<<< Facebook — Rank 1–344 — N Handles >>>
(...)
OR
…
))
```

In [5]:
total_handles = sum(s.count('author:"') for s in all_blocks)
header = f"<<< Politik-Query — {total_handles} Handles in {len(all_blocks)} Blöcken >>>"

body = "\nOR\n".join(all_blocks)
query = f"{header}\n({LANGUAGE_FILTER} AND (\n{body}\n))\n"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(query)

size = os.path.getsize(OUTPUT_FILE)
print(f"Datei:           {OUTPUT_FILE}")
print(f"Größe:           {size:,} bytes  ({size / 1024:.1f} KiB)")
print(f"Handles gesamt:  {total_handles}")
print(f"Blöcke:          {len(all_blocks)}")

if size > 100_000:
    over = size - 100_000
    print(
        f"\n⚠️  {over:,} Zeichen über dem 100k-Limit. "
        f"Unten stehen die Blöcke in absteigender Reihenfolge; einfach die "
        f"letzten (= höchste Ranks) aus der TXT-Datei löschen, bis's passt."
    )

Datei:           /Users/zorbeyozcan/Projekte/query_printer/output/queries/politics_query.txt
Größe:           171,830 bytes  (167.8 KiB)
Handles gesamt:  6365
Blöcke:          23

⚠️  71,830 Zeichen über dem 100k-Limit. Unten stehen die Blöcke in absteigender Reihenfolge; einfach die letzten (= höchste Ranks) aus der TXT-Datei löschen, bis's passt.


## 5. Preview — Block-Struktur + Größe pro Block

In [6]:
print(f"{'#':>3}  {'Block-Größe':>12}  {'Kumul.':>12}  Kommentar")
print("-" * 100)
cum = 0
for i, b in enumerate(all_blocks, 1):
    # +4 für "\nOR\n" das zwischen Blöcken steht
    sep = 4 if i > 1 else 0
    sz = len(b) + sep
    cum += sz
    comment = b.split("\n", 1)[0]
    limit_flag = "  ← 100k erreicht" if cum > 100_000 and cum - sz <= 100_000 else ""
    print(f"{i:>3}  {sz:>12,}  {cum:>12,}  {comment}{limit_flag}")

  #   Block-Größe        Kumul.  Kommentar
----------------------------------------------------------------------------------------------------
  1        43,517        43,517  <<< X (Twitter) — 1781 Handles >>>
  2        28,616        72,133  <<< Facebook — ohne Ranking — 1011 Handles >>>
  3         2,404        74,537  <<< Facebook — Rank 1–344 — 87 Handles >>>
  4         2,228        76,765  <<< Facebook — Rank 345–688 — 78 Handles >>>
  5         1,889        78,654  <<< Facebook — Rank 689–1032 — 65 Handles >>>
  6         5,080        83,734  <<< Facebook — Rank 1033–1376 — 183 Handles >>>
  7         5,543        89,277  <<< Facebook — Rank 1377–1720 — 187 Handles >>>
  8         2,926        92,203  <<< Facebook — Rank 1721–2064 — 99 Handles >>>
  9         4,051        96,254  <<< Facebook — Rank 2065–2408 — 144 Handles >>>
 10         4,296       100,550  <<< Facebook — Rank 2409–2752 — 152 Handles >>>  ← 100k erreicht
 11         4,643       105,193  <<< Facebook — Rank 2